# 🚀 ذكاء اصطناعي لاستخراج توقيت القرآن (Wav2Vec2)

هذا الملف مخصص لتشغيل نموذج `Wav2Vec2` للغة العربية لاستخراج أوقات بداية ونهاية كل كلمة أو آية، ثم حفظها في قاعدة بيانات **Supabase** الخاصة بك.

### ⚡ طريقة التشغيل:
1. اذهب للقائمة العلوية واضغط على `Runtime` ثم `Change runtime type`.
2. تأكد أن `Hardware accelerator` محدد على **T4 GPU** ليكون سريعاً جداً كالبرق.
3. اضغط على أيقونة **التشغيل (▶)** بجانب كل مربع كود بالأسفل بالترتيب.

In [ ]:
# 1. تثبيت المكتبات اللازمة للذكاء الاصطناعي
!pip install -q transformers torchaudio supabase librosa requests

In [ ]:
import os
from supabase import create_client, Client
from transformers import pipeline
import requests
import json

# ==========================================
# 2. إعداد قاعدة البيانات (Supabase)
# ==========================================
SUPABASE_URL = "https://zqnwhrvwovwdakuuxpkr.supabase.co"
SUPABASE_KEY = "sb_publishable_kjhsyZVWYqoy-7sX9jJx2A_TyELp9eD"
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

print("✅ تم الاتصال بقاعدة البيانات بنجاح!")

In [ ]:
# ==========================================
# 3. إعداد نموذج الذكاء الاصطناعي (Wav2Vec2)
# ==========================================
print("جاري تحميل النموذج (قد يستغرق 30 ثانية في المرة الأولى)...")
asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model="elgeish/wav2vec2-base-ar-vocab",
    return_timestamps="word",
    device=0 # استخدام كرت الشاشة (GPU)
)
print("✅ تم تحميل نموذج الذكاء الاصطناعي بنجاح!")

In [ ]:
# ==========================================
# 4. دالة المعالجة واستخراج الأوقات
# ==========================================
def process_surah(surah_number):
    print(f"\nجاري معالجة سورة رقم {surah_number}...")
    
    # تحميل الصوت من Hugging Face (يجب أن يكون عام Public)
    audio_url = f"https://huggingface.co/datasets/hammoualiyoucef20/quran-audio/resolve/main/{surah_number}.mp3"
    
    response = requests.get(audio_url)
    if response.status_code != 200:
        print(f"❌ خطأ: لم يتم العثور على الملف {surah_number}.mp3 أو أن المساحة الخاصة بك لا تزال Private!")
        return
        
    with open("temp.mp3", "wb") as f:
        f.write(response.content)
        
    print("🧠 جاري استخراج التوقيت الزمني الدقيق بالذكاء الاصطناعي...")
    result = asr_pipeline("temp.mp3")
    chunks = result.get("chunks", [])
    
    boxes = []
    for chunk in chunks:
        boxes.append({
            "word": chunk["text"],
            "start": chunk["timestamp"][0],
            "end": chunk["timestamp"][1]
        })
        
    print(f"✅ تم استخراج {len(boxes)} كلمة بالتوقيت!")
    
    # الحفظ في قاعدة البيانات
    try:
        supabase.table("ayah_coordinates").upsert({
            "page_src": f"{surah_number}.mp3",
            "boxes": boxes
        }).execute()
        print(f"🚀 تم رفع توقيت السورة إلى قاعدة بياناتك بنجاح! الموقع جاهز لاستخدامها.")
    except Exception as e:
        print(f"❌ خطأ أثناء الرفع: {e}")

In [ ]:
# ==========================================
# 5. التشغيل!
# ==========================================
# هنا ضع رقم السورة أو الصوت الذي رفعته (مثلاً 2)
process_surah(2)